# DEMO - Visual anomaly detection on LUMI

## Evaluating and fine tuning AnomalyCLIP on LUMI

Phase 0: Setting up dataset and environment.

Phase 1: Evaluate the VisA checkpoint on a held-out Real-IAD split. 

Phase 2: Fine-tune on Real-IAD, re-evaluate, compare.

### Phase 0: Setting up dataset and environment

import argparse
import os
import sys
import time
import torch
import pandas as pd

from datasets import Dataset

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT")
USER = os.getenv("SLURM_JOB_USER")

print(f"Billing project:  {SLURM_JOB_ACCOUNT}")
print(f"Username:         {USER}")

In [ ]:
input_model = "AnomalyCLIP"
# All outputs are saved next to the notebooks
output_path = os.path.join(os.getcwd(), "ft_model")
model_output_name = f"{input_model.split('/')[-1]}_finetuned"

num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

# Output model 
output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

print(f"Model will be saved to: {merged_output_dir}")

Cloning the AnomalyCLIP repo

In [ ]:
!git clone https://github.com/zqhang/AnomalyCLIP.git

#### Dataset inspection - subset of Real-IAD dataset

The given dataset is a subset of the Real-IAD dataset including 3 product image sets: audiojack, toothbrush and zipper. 

In [ ]:
# processing dataset - ALREADY DONE BEFORE THE DEMO

import json, random, sys

root, out_eval, out_ft = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])
max_per_cls = 400
random.seed(0)

# TODO: CHECK THE PATH
root = /Users/mainguye/Personal/Working/demo/visual-anomaly-detection-demo/dataset/realiad_1024
out_eval = /Users/mainguye/Personal/Working/demo/visual-anomaly-detection-demo/dataset/realiad_1024/data/out_eval
out_ft = /Users/mainguye/Personal/Working/demo/visual-anomaly-detection-demo/dataset/realiad_1024/data/ft



In [ ]:
def resolve(rel, cls, prefix):
    """Return the first existing candidate path for an image/mask path from the json."""
    rel = str(rel).lstrip("/")
    for p in (root / rel, root / cls / rel, root / prefix / rel if prefix else None, root / cls / Path(rel).name):
        if p is not None and p.exists():
            return str(p)
    return str(root / rel)

In [ ]:
ev, ft = {}, {}
for jf in sorted((root / "realiad_jsons_sv").glob("*.json")):
    d = json.load(open(jf))
    cls = jf.stem
    prefix = d.get("meta", {}).get("prefix", "") if isinstance(d.get("meta"), dict) else ""
    items = list(d["test"])
    random.shuffle(items)
    recs = []
    for it in items[:max_per_cls]:
        ok = it["anomaly_class"] == "OK"
        mask = it.get("mask_path") or ""
        recs.append(dict(
            img_path=resolve(it["image_path"], cls, prefix),
            mask_path="" if (ok or not mask) else resolve(mask, cls, prefix),
            cls_name=cls, specie_name="" if ok else it["anomaly_class"], anomaly=0 if ok else 1))
    h = len(recs) // 2
    ft[cls], ev[cls] = recs[:h], recs[h:]
for out, test in ((out_eval, ev), (out_ft, ft)):
    out.mkdir(parents=True, exist_ok=True)
    json.dump({"train": {}, "test": test}, open(out / "meta.json", "w"))
    print(out, {k: len(v) for k, v in test.items()})


In [ ]:
# Inspecting the dataset

# TODO: CODE FOR INSPECTION

In [ ]:
# Dataset splitting - An example

# TODO: CODE FOR SPLITTING DATASET

### Phase 1: Evaluate the VisA checkpoint on a held-out REAL-IAE split

In this demo, we will use the given script from AnomalyCLIP's authors to evaluate the model before fine-tuning

In [ ]:
python test.py --dataset realiad --data_path $DATA --save_path $RESULTS_DIR/$TAG \
    --checkpoint_path $CKPT --features_list 24 --image_size 518 --depth 9 --n_ctx 12 --t_n_ctx 4 --sigma 4
